# 04 - Modèle avec contexte

Ce notebook entraîne un modèle NCF qui intègre des features contextuelles extraites de MovieLens 100K.

L'objectif est de comparer une architecture baseline sans contexte avec un modèle qui utilise l'heure, le jour de la semaine, et des features de session.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, encode_ids
from context_features import extract_all_context_features
from models import NCFContext
from metrics import hit_rate, ndcg, mrr

print('Python:', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy :', np.__version__)
print('torch :', torch.__version__)


Python: 3.12.13
pandas : 3.0.3
numpy : 2.4.6
torch : 2.12.0+cu130


## Chargement et extraction des features de contexte

Nous utilisons MovieLens 100K en chargeant le fichier brut et en extrayant les features temporelles et de session.


In [2]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
print('RAW_ROOT =', RAW_ROOT)

df = load_movielens_100k(RAW_ROOT)
print('raw shape:', df.shape)

df = encode_ids(df)
df = extract_all_context_features(df)

print('processed shape:', df.shape)
print(df.columns.tolist())
print(df[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_sin', 'session_length', 'session_position_norm']].head())


RAW_ROOT = /home/mrtds/Documents/my_projects/context-aware-recsys/data/raw/movielens/ml-100k
raw shape: (100000, 4)
processed shape: (100000, 19)
['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
   user_id_encoded  item_id_encoded  hour_of_day  dow_sin  session_length  \
0                0              167           21      0.0              20   
1                0              171           21      0.0              20   
2                0              164           21      0.0              20   
3                0              155           21      0.0              20   
4                0              195           22      0.0              20   

   session_position_norm  
0               0.000000  
1               0.05

### Features contextuelles utilisées

Nous utilisons un ensemble de features continues extraites pour représenter l'heure, le jour de la semaine, et le comportement de session.


In [3]:
context_cols = [
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend',
    'time_since_last_interaction_log',
    'session_length',
    'session_position_norm'
]
print('Context columns:', context_cols)
print(df[context_cols].head())


Context columns: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'time_since_last_interaction_log', 'session_length', 'session_position_norm']
   hour_sin  hour_cos  dow_sin  dow_cos  is_weekend  \
0 -0.707107  0.707107      0.0      1.0           0   
1 -0.707107  0.707107      0.0      1.0           0   
2 -0.707107  0.707107      0.0      1.0           0   
3 -0.707107  0.707107      0.0      1.0           0   
4 -0.500000  0.866025      0.0      1.0           0   

   time_since_last_interaction_log  session_length  session_position_norm  
0                         0.000000              20               0.000000  
1                         0.000000              20               0.052632  
2                         3.713572              20               0.105263  
3                         3.663562              20               0.157895  
4                         4.804021              20               0.210526  


## Préparation des labels

Nous convertissons les notes en labels binaires pour entraîner un modèle de ranking simple.


In [4]:
df['label'] = (df['rating'] >= 4).astype(int)
print(df['label'].value_counts(normalize=True))

n_users = df['user_id_encoded'].nunique()
n_items = df['item_id_encoded'].nunique()
context_dim = len(context_cols)

print('Users:', n_users, 'Items:', n_items, 'Context dim:', context_dim)


label
1    0.55375
0    0.44625
Name: proportion, dtype: float64
Users: 943 Items: 1682 Context dim: 8


## Split train/test

Nous utilisons un split aléatoire en 80/20. On pourra remplacer par un split temporel spécifique plus tard.


In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)


train shape: (80000, 20)
test shape: (20000, 20)


## Dataset PyTorch avec contexte

Le dataset renvoie l'item, l'utilisateur, le contexte et le label.


In [6]:
class ContextMovielensDataset(Dataset):
    def __init__(self, df, context_columns):
        self.user_ids = torch.tensor(df['user_id_encoded'].values, dtype=torch.long)
        self.item_ids = torch.tensor(df['item_id_encoded'].values, dtype=torch.long)
        self.contexts = torch.tensor(df[context_columns].values, dtype=torch.float32)
        self.labels = torch.tensor(df['label'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.contexts[idx], self.labels[idx]

train_dataset = ContextMovielensDataset(train_df, context_cols)
test_dataset = ContextMovielensDataset(test_df, context_cols)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print('Train batches:', len(train_loader))
print('Test batches:', len(test_loader))


Train batches: 313
Test batches: 79


## Construction du modèle contextuel

Nous utilisons `NCFContext` avec une fusion par concaténation. Cet exemple peut être étendu ultérieurement avec `film` ou `attention`.


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = NCFContext(
    num_users=n_users,
    num_items=n_items,
    context_dim=context_dim,
    embed_dim=32,
    mlp_layers=[64, 32, 16],
    context_embed_dim=32,
    dropout=0.2,
    fusion_type='concat'
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

print(model)


Device: cpu
NCFContext(
  (ncf_core): NCF(
    (embedding_user_gmf): Embedding(943, 32)
    (embedding_item_gmf): Embedding(1682, 32)
    (embedding_user_mlp): Embedding(943, 32)
    (embedding_item_mlp): Embedding(1682, 32)
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=64, out_features=32, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.2, inplace=False)
      (6): Linear(in_features=32, out_features=16, bias=True)
      (7): ReLU()
      (8): Dropout(p=0.2, inplace=False)
    )
    (output_layer): Linear(in_features=48, out_features=1, bias=True)
  )
  (context_encoder): ContextEncoder(
    (net): Sequential(
      (0): Linear(in_features=8, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=64, out_features=32, bias=True)
      (4): ReLU()
    )
  )
  (final_mlp): Sequential(
    (0): Lin

## Entraînement

Nous entraînons le modèle avec une boucle simple et surveillons la perte d'entraînement.


In [8]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for user_batch, item_batch, context_batch, labels in loader:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        context_batch = context_batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(user_batch, item_batch, context_batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)

epochs = 8
history = []
for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    history.append(loss)
    print(f'Epoch {epoch}/{epochs} - train loss: {loss:.4f}')


Epoch 1/8 - train loss: 0.6236
Epoch 2/8 - train loss: 0.5634
Epoch 3/8 - train loss: 0.5221
Epoch 4/8 - train loss: 0.4004
Epoch 5/8 - train loss: 0.2923
Epoch 6/8 - train loss: 0.2274
Epoch 7/8 - train loss: 0.1819
Epoch 8/8 - train loss: 0.1475


## Évaluation ranking

Nous évaluons sur un sous-ensemble de test pour maîtriser le temps d'inférence.


In [10]:
eval_df = test_df.sample(n=min(1000, len(test_df)), random_state=42).reset_index(drop=True)
all_item_ids = torch.arange(n_items, dtype=torch.long, device=device)

def evaluate_ranking(model, df, k=10):
    model.eval()
    hits, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for _, row in df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            context = torch.tensor(row[context_cols].values.astype(np.float32), dtype=torch.float32, device=device).unsqueeze(0)
            user_batch = user_id.expand(n_items)
            context_batch = context.expand(n_items, context_dim)
            logits = model(user_batch, all_item_ids, context_batch).cpu().numpy()
            ranking = np.argsort(-logits)
            hits.append(int(item_id in ranking[:k]))
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)

hit10, ndcg10, mrr_score = evaluate_ranking(model, eval_df, k=10)
print(f'Hit Rate@10: {hit10:.4f}')
print(f'NDCG@10: {ndcg10:.4f}')
print(f'MRR: {mrr_score:.4f}')

Hit Rate@10: 0.0210
NDCG@10: 0.0096
MRR: 0.0109


## Sauvegarde du modèle contextuel

Nous sauvegardons les poids pour une comparaison avec le baseline.


In [11]:
RESULTS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = RESULTS_DIR / 'context_ncf_ml100k.pt'
torch.save(model.state_dict(), MODEL_PATH)
print('Saved context model to', MODEL_PATH)


Saved context model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/context_ncf_ml100k.pt


## Résumé

Ce notebook entraîne un NCF enrichi par des features contextuelles sur MovieLens 100K.
La comparaison avec le baseline permettra d'évaluer la valeur ajoutée des données temporelles et de session.
